## Spot filtrr

The code in this notebook takes the raw spot counts and metrics generated by spot-findr.ipynb and the cell metrics generated by img-explorer.ipynb and aims to filter true spots for each true cell

In [ ]:
# Import necessary packages

import os
import glob
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import logging

import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import plotnine as p9 
import pandas as pd
from skimage import feature, io
from scipy.stats import norm
from scipy.ndimage import label, find_objects

# Make path for scripts relative to the working directory
sys.path.insert(0, '../src')  # Adjust the path as needed

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

We have cells with a given number of spots. Let's first see what's the distribution of spots for each experimental group

In [ ]:
# Plot the number of spots found for all ROIs in all images, 
# separated by experimental group and channel

# Define a dictionary of channel colors
channel_colors = {1: 'blue', 2: 'green', 3: 'magenta', 4: 'cyan'}

# Define some list of values to filter by

# Reorder channels so that Channel 2 (GFP) is plotted over Channel 3 (FISH)
#spot_filter_counts['channel'] = pd.Categorical(spot_filter_counts['channel'], categories=[3, 2], ordered=True)

#my_spots = spot_counts[spot_counts['channel'].isin([2,3])]
my_spots = spot_counts[spot_counts['experiment'] == experiment_list[0]]

# Determine the largest number of spots to set the X axis of the plot
#max_spot_count = spot_df['spot_count'].max()
max_spot_count = 8

# Define the plot with overlaid histograms for each channel
spot_histogram_raw = (
    p9.ggplot(my_spots, 
              p9.aes(x = 'spot_count', 
                     fill = 'factor(channel)')) +  # Set the data and x-axis aesthetic
    p9.geom_histogram(binwidth = 1, color = 'black', 
                      alpha = 0.5, position = 'identity') +  # Overlaid histograms
    p9.labs(title = 'Spot Counts per Cell', 
            x = 'Spots', y = 'Frequency', fill = 'Channel') +  # Add labels
    #p9.facet_wrap(' ~ channel + group', scales = "free_y", nrow = 2) +
    p9.facet_wrap(' ~ plasmid', scales = 'free_y') +
    p9.scale_fill_manual(values = channel_colors) +  # Apply the custom channel colors
    p9.scale_x_continuous(limits = [0, max_spot_count+1], breaks = range(0, max_spot_count + 1, 2)) + 
    p9.theme_bw() +  # Apply theme
    p9.theme(figure_size = (12, 6),
             subplots_adjust = {'wspace': 0.4},  # Adjust space between plots
             aspect_ratio = 1,
             panel_grid = p9.element_blank(), 
             panel_border = p9.element_rect(color='black', linewidth=1), 
             axis_ticks = p9.element_line(color='black', linewidth=0.5), 
             plot_title = p9.element_text(size=12, color='black', face='bold'),
             axis_text = p9.element_text(size=8, color='black'), 
             axis_title = p9.element_text(size=10, color='black', face='bold'), 
             legend_text = p9.element_text(size=8, color='black'), 
             legend_title = p9.element_text(size=10, color='black', face='bold'),
             legend_position = 'none',
             strip_background = p9.element_blank(), 
             strip_text = p9.element_text(size=6, color='black'))
)

# Display the plot
print(spot_histogram_raw)

In [ ]:
# Assess spot intensity distribution

# Pick images to plot, by index
selected_indices = list(range(4))

# Pick the channel to plot
c = 2

# or by name
# base_name_to_plot = "example_name"

# 1. Filter spot_metrics such that it includes images only with spots for channel c.

# Filter spot_counts to only include rows for channel c and a positive spot count.
images_with_spots = spot_counts[(spot_counts['spot_count'] > 0) & (spot_counts['channel'] == c)]['base_name'].unique()
# Filter spot_metrics to include only these images.
spot_metrics_yespots = spot_metrics[spot_metrics['base_name'].isin(images_with_spots)]

# 2. Select images to plot from the filtered data.

# Here, we'll use a selection of indices from the unique base_names in spot_metrics_yespots.
unique_base_names = spot_metrics_yespots['base_name'].unique()
images_to_plot = [unique_base_names[i] for i in selected_indices if i < len(unique_base_names)]

# Define the grid dimensions.
n_images = len(images_to_plot)
ncols = 4
nrows = int(np.ceil(n_images / ncols))

# Create subplots grid.
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4), squeeze=False)

# Loop over each selected image and plot it.
for idx, base_name in enumerate(images_to_plot):
    ax = axes[idx // ncols, idx % ncols]
    
    # Filter the DataFrame for the current image.
    subset = spot_metrics_yespots[(spot_metrics_yespots['base_name'] == base_name) &
                                  (spot_metrics_yespots['channel'] == c)]
    
    print(f"{base_name} (Channel {c}): {len(subset)} spots")
    
    # Create a scatter plot: ROI index on x-axis, spot intensity on y-axis.
    ax.scatter(subset['ROI'], subset['intensity'], alpha=0.7, edgecolor='black')
    ax.set_title(f"Image: {base_name}\nChannel: {c}")
    ax.set_xlabel("ROI index")
    ax.set_ylabel("Spot Intensity")
    
    # Set x-axis ticks to show unique integer ROI values.
    unique_rois = np.sort(subset['ROI'].unique())
    ax.set_xticks(unique_rois)

# Hide any unused subplots in the grid.
total_subplots = nrows * ncols
for j in range(idx + 1, total_subplots):
    fig.delaxes(axes[j // ncols, j % ncols])

plt.tight_layout()
plt.show()

In [ ]:
# Calculate the relative intensity for each ROI:

# Compute the maximum intensity for each group of base_name and ROI
spot_metrics['max_intensity'] = spot_metrics.groupby(['base_name', 'ROI', 'channel'])['intensity'].transform('max')

# Create the new variable 'relative_intensity' by dividing each spot's intensity by the group's maximum intensity
spot_metrics['rel_intensity'] = spot_metrics['intensity'] / spot_metrics['max_intensity']

# Optionally, drop the intermediate column if you don't need it anymore
spot_metrics.drop(columns = 'max_intensity', inplace = True)

# Inspect the updated DataFrame
print(spot_metrics[['base_name', 'ROI', 'intensity', 'rel_intensity']].head())

In [ ]:
# Assess relative spot intensity distribution

# Pick images to plot, by index
selected_indices = list(range(8))
#selected_indices = [5]

# Pick the channel to plot
c = 2

# Define the potential cutoff for spurious spots, 
# as a fraction of the brigthest spot in the cell
cutoff = 0.7

# or by name
# base_name_to_plot = "example_name"

# 1. Filter spot_metrics such that it includes images only with spots for channel c.

# Filter spot_counts to only include rows for channel c and a positive spot count.
images_with_spots = spot_counts[(spot_counts['spot_count'] > 0) & (spot_counts['channel'] == c)]['base_name'].unique()
# Filter spot_metrics to include only these images.
spot_metrics_yespots = spot_metrics[spot_metrics['base_name'].isin(images_with_spots)]

# 2. Select images to plot from the filtered data.

# Here, we'll use a selection of indices from the unique base_names in spot_metrics_yespots.
unique_base_names = spot_metrics_yespots['base_name'].unique()
images_to_plot = [unique_base_names[i] for i in selected_indices if i < len(unique_base_names)]

# Define the grid dimensions.
n_images = len(images_to_plot)
ncols = 4
nrows = int(np.ceil(n_images / ncols))

# Create subplots grid.
#fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4), squeeze=False)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4), squeeze=False)


# Loop over each selected image and plot it.
for idx, base_name in enumerate(images_to_plot):
    ax = axes[idx // ncols, idx % ncols]
    
    # Filter the DataFrame for the current image.
    subset = spot_metrics_yespots[(spot_metrics_yespots['base_name'] == base_name) &
                                  (spot_metrics_yespots['channel'] == c)]
    
    print(f"{base_name} (Channel {c}): {len(subset)} spots")
    
    # Create a scatter plot: ROI index on x-axis, spot intensity on y-axis.
    ax.scatter(subset['ROI'], subset['rel_intensity'], alpha=0.7, edgecolor='black')
    ax.set_title(f"Image: {base_name}\nChannel: {c}")
    ax.axhline(y=cutoff, color='k', linestyle='-')
    ax.set_xlabel("ROI index")
    ax.set_ylabel("Spot relative intensity")
    
    # Set x-axis ticks to show unique integer ROI values.
    unique_rois = np.sort(subset['ROI'].unique())
    ax.set_xticks(unique_rois)

# Hide any unused subplots in the grid.
total_subplots = nrows * ncols
for j in range(idx + 1, total_subplots):
    fig.delaxes(axes[j // ncols, j % ncols])

plt.tight_layout()
plt.axhline(y=cutoff, color='k', linestyle='-')
plt.show()

In [ ]:
# Filter spots based on the cutoff defined above:
spot_filter_metrics = spot_metrics[spot_metrics['rel_intensity'] >= cutoff]
# generate a new counts table post-filtering
spot_filter_counts = spot_filter_metrics.groupby(['experiment', 'base_name', 'ROI', 'channel']).size().reset_index(name = 'spot_count')
# and incorporate the experimental metadata, as done before:
spot_filter_counts = pd.merge(spot_filter_counts, 
                       exp_groups, 
                       on = ["experiment", "base_name"], how = 'left')

In [ ]:
# Make a histogram showing the number of spots after filtering, as done earlier

#my_spots = spot_filter_counts[spot_counts['channel'].isin([2,3])]
my_spots = spot_filter_counts[spot_filter_counts['experiment'] == experiment_list[1]]


# Determine the largest number of spots to set the X axis of the plot
#max_spot_count = spot_df['spot_count'].max()
max_spot_count = 8

# Define the plot with overlaid histograms for each channel
spot_histogram_filter = (
    p9.ggplot(my_spots, 
              p9.aes(x = 'spot_count', 
                     fill = 'factor(channel)')) +  # Set the data and x-axis aesthetic
    p9.geom_histogram(binwidth = 1, color = 'black', 
                      alpha = 0.5, position = 'identity') +  # Overlaid histograms
    p9.labs(title = 'Spot Counts per Cell', 
            x = 'Spots', y = 'Frequency', fill = 'Channel') +  # Add labels
    #p9.facet_wrap(' ~ channel + group', scales = "free_y", nrow = 2) +
    p9.facet_wrap(' ~ plasmid', scales = 'free_y') +
    p9.scale_fill_manual(values = channel_colors) +  # Apply the custom channel colors
    p9.scale_x_continuous(limits = [0, max_spot_count+1], breaks = range(0, max_spot_count + 1, 2)) + 
    p9.theme_bw() +  # Apply theme
    p9.theme(figure_size = (12, 6),
             subplots_adjust = {'wspace': 0.4},  # Adjust space between plots
             aspect_ratio = 1,
             panel_grid = p9.element_blank(), 
             panel_border = p9.element_rect(color='black', linewidth=1), 
             axis_ticks = p9.element_line(color='black', linewidth=0.5), 
             plot_title = p9.element_text(size=12, color='black', face='bold'),
             axis_text = p9.element_text(size=8, color='black'), 
             axis_title = p9.element_text(size=10, color='black', face='bold'), 
             legend_text = p9.element_text(size=8, color='black'), 
             legend_title = p9.element_text(size=10, color='black', face='bold'),
             legend_position = 'none',
             strip_background = p9.element_blank(), 
             strip_text = p9.element_text(size=6, color='black'))
)

# Display the plot
print(spot_histogram_filter)

In [ ]:
# Optional: save the plot to "plots" inside "repo_directory"

# Set current experiment; you can loop through experiment_list if needed.
current_experiment = experiment_list[0]  # or any experiment of interest

# Define the folder path: repo_directory/plots/<experiment>
plots_folder = os.path.join(repo_directory, "plots", current_experiment)
os.makedirs(plots_folder, exist_ok = True)  # Create the folder if it doesn't exist

# Give the plot a filename
plot_filename = os.path.join(plots_folder, "spot_histogram_unfiltered.png")

# Draw the Plotnine plot to obtain a Matplotlib figure
fig = spot_histogram_raw.draw()

# Save the figure as a PNG file
fig.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.close(fig)  # Close the figure after saving

print(f"Plot saved as PNG to {plot_filename}")

# Give the plot a filename
plot_filename = os.path.join(plots_folder, "spot_histogram_filtered.png")

# Draw the Plotnine plot to obtain a Matplotlib figure
fig = spot_histogram_filter.draw()

# Save the figure as a PNG file
fig.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.close(fig)  # Close the figure after saving

print(f"Plot saved as PNG to {plot_filename}")

#### **The cells below have bits and scraps of old code that needs to be properly integrated above or removed**

In [ ]:
# Read in the cell classification generated by me with img-explorer.ipynb
cell_clx = pd.read_csv(os.path.join(repo_directory, "results/", "Cen-ZFs_select_HCT116_all_class.csv"))
cell_clx.rename(columns = {'filename': 'base_name', 
                           'ROI_index': 'ROI'}, inplace = True)

# And also the data frame with fluorescent intensities generated with img-explorer.ipynb
cell_metrix = pd.read_csv(os.path.join(repo_directory, "results/", "Cen-ZFs_select_HCT116_all.csv"))
cell_metrix.rename(columns = {'filename': 'base_name',
                              'ROI_index': 'ROI'}, inplace = True)

# Adjust the channel numbers in cell_metrix so they match spot_metrics
cell_metrix['channel'] = cell_metrix['channel'] + 1

In [ ]:
# Merge spot_counts with my cell classification
spot_counts = pd.merge(spot_counts, 
                   cell_clx[['plasmid', 'experiment', 'base_name', 'ROI', 'tx_status_thr', 'tx_status_gmm', 'cor_ch1_log10_median']], 
                   on = ['base_name', 'experiment', 'ROI'], how = 'inner')

In [ ]:
# Incorporate ROI median intensity into spots_metrics data frame
spot_metrics = pd.merge(
    spot_metrics,
    cell_metrix[['plasmid', 'experiment', 'base_name', 'ROI', 'channel', 'median']],
    on = ['base_name', 'experiment', 'ROI', 'channel'], 
    how = 'inner'
)

In [ ]:
# Calculate the signal-to-background ratio as the spot intensity divided by ROI median intensity
spot_metrics['sbr'] = spot_metrics['intensity'] / spot_metrics['median']

In [ ]:
# Visualize the distribution of SBR values as a histogram

# Drop any NaN values that might have been created due to division by zero or missing data.
sbr_values = spot_metrics['sbr'].dropna()

# Compute statistics: median, 25th percentile (Q1) and 75th percentile (Q3)
median_val = np.median(sbr_values)
q1, q3 = np.percentile(sbr_values, [25, 75])

# Create histogram
plt.figure(figsize=(8, 6))
plt.hist(sbr_values, bins=50, edgecolor='black', alpha=0.7)

# Add a vertical dashed line at Q1
plt.axvline(q1, color='red', linestyle='dashed', linewidth=1.5, label=f"Q1: {q1:.2f}")

plt.xlabel('Signal to Background Ratio (sbr)')
plt.ylabel('Frequency')
plt.title('Distribution of sbr values')

# Add text box with median and quartiles
stats_text = f"Median: {median_val:.2f}\nQ1 (25th percentile): {q1:.2f}\nQ3 (75th percentile): {q3:.2f}"
plt.text(0.95, 0.95, stats_text, transform=plt.gca().transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.7))

plt.show()

In [ ]:
# Visualize the distribution of SBR values as a histogram, by plasmid

# Get unique plasmid values
plasmids = spot_metrics['plasmid'].unique()
num_plasmids = len(plasmids)

# Define the grid layout (adjust ncols as needed)
ncols = 4
nrows = int(np.ceil(num_plasmids / ncols))

# Create the figure and axes array
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows))
axes = axes.flatten()  # Flatten in case we have multiple rows/columns

for i, plasmid in enumerate(plasmids):
    ax = axes[i]
    # Filter the data for the current plasmid and drop NaN sbr values
    plasmid_data = spot_metrics.loc[spot_metrics['plasmid'] == plasmid, 'sbr'].dropna()
    
    # Compute statistics for the current plasmid
    median_val = np.median(plasmid_data)
    q1, q3 = np.percentile(plasmid_data, [25, 75])
    
    # Plot the histogram for the current plasmid
    ax.hist(plasmid_data, bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(f'Plasmid: {plasmid}')
    ax.set_xlabel('Signal to Background Ratio (sbr)')
    ax.set_ylabel('Frequency')
    
    # Add a text box with statistics in the top-right corner
    stats_text = f"Median: {median_val:.2f}\nQ1: {q1:.2f}\nQ3: {q3:.2f}"
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.7))

# Remove any unused subplots if the total number of plasmids is less than nrows*ncols
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Filter spots based on the calculated SBR, **for each plasmid**

# Step 1: Filter spot_metrics within each plasmid group
filtered_spots = spot_metrics.groupby('plasmid').apply(
    lambda group: group[group['sbr'] >= np.percentile(group['sbr'].dropna(), 25)]
).reset_index(drop=True)

# Step 2: Group the filtered data by the specified columns and count the number of spots per group.
spot_filter_counts = filtered_spots.groupby(['experiment', 'base_name', 'ROI', 'channel']).size().reset_index(name='spot_count')

# Merge spot_counts with my cell classification
spot_filter_counts = pd.merge(
    cell_clx[['plasmid', 'experiment', 'base_name', 'ROI', 'tx_status_thr', 'tx_status_gmm', 'cor_ch1_log10_median']], 
    spot_filter_counts[['experiment', 'base_name', 'ROI', 'spot_count']], 
    on = ['base_name', 'experiment', 'ROI'], 
    how = 'left'
)

spot_filter_counts['spot_count'] = spot_filter_counts['spot_count'].fillna(0).astype(int)

# Visualize the result
print(spot_filter_counts.head(10))

In [ ]:
# Filter spots based on the calculated SBR, **globally**

# Step 1: Filter spot_metrics to include only spots with sbr >= 25th percentile.
# Calculate the 25th percentile (Q1) of sbr values
q1 = np.percentile(spot_metrics['sbr'].dropna(), 25)
filtered_spots = spot_metrics[spot_metrics['sbr'] >= q1].copy() 

# Step 2: Group the filtered data by the specified columns and count the number of spots per group.
spot_filter_counts = filtered_spots.groupby(['experiment', 'base_name', 'ROI', 'channel']).size().reset_index(name='spot_count')

# Merge spot_counts with my cell classification
spot_filter_counts = pd.merge(
    cell_clx[['plasmid', 'experiment', 'base_name', 'ROI', 'tx_status_thr', 'tx_status_gmm', 'cor_ch1_log10_median']], 
    spot_filter_counts[['experiment', 'base_name', 'ROI', 'spot_count']], 
    on = ['base_name', 'experiment', 'ROI'], how = 'left'
)

spot_filter_counts['spot_count'] = spot_filter_counts['spot_count'].fillna(0).astype(int)

# Visualize the result
print(spot_filter_counts.head(10))